# 03B — Historical feature quality experiments

This notebook tests whether partially observed historical days can supply reliable predictor values. It starts from genuine 96/96 household-days, then removes slots using missing-slot patterns observed on 95/94/93/92-of-96 days. It compares each masked daily statistic with its complete-day value.

The analysis is separate from 03A: 03A tests target-label reliability; this notebook tests historical predictor reliability. It does not train models or use model performance to select a quality rule. Feature-specific candidate cutoffs are screening results for researcher review, not a frozen policy.

The primary features are daily peak, mean, active-import ratio, load factor, interval P95, and peak hour. Daily standard deviation is included as an additional profile statistic. Daily energy is audited separately because an incomplete-day sum is not extrapolated.


In [ ]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import json
import math
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.5f}")

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent
if not (PROJECT_ROOT / "data").exists():
    raise RuntimeError(f"Could not locate project root from {CURRENT_DIR}")

DATA_DIR = PROJECT_ROOT / "data"
INTERVAL_PATH = DATA_DIR / "processed" / "01_preprocessed_lp_interval_data.csv"
PREPROCESSING_MANIFEST_PATH = DATA_DIR / "processed" / "01_preprocessing_manifest.json"
COVERAGE_PATH = DATA_DIR / "interim" / "01_household_day_coverage_audit.csv"
COMPLETE_DAY_PATH = PROJECT_ROOT / "reports" / "tables" / "02_interval_eda" / "02_05_complete_day_shape_metrics.csv"
EDA_MANIFEST_PATH = PROJECT_ROOT / "reports" / "tables" / "02_interval_eda" / "02_eda_manifest.json"
TARGET_CONFIG_PATH = PROJECT_ROOT / "reports" / "tables" / "03a_target_design_experiments" / "03a_selected_target_config.json"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables" / "03b_historical_feature_quality_experiments"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures" / "03b_historical_feature_quality_experiments"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = TABLE_DIR / "03b_feature_masking_summary.csv"
POLICY_PATH = TABLE_DIR / "03b_feature_quality_policy_candidates.csv"
ENERGY_PATH = TABLE_DIR / "03b_energy_redundancy_audit.csv"
VALIDATION_PATH = TABLE_DIR / "03b_validation_checks.csv"
MANIFEST_PATH = TABLE_DIR / "03b_run_manifest.json"
CONFIG_PATH = TABLE_DIR / "03b_selected_feature_quality_config.json"
BOX_FIGURE_PATH = FIGURE_DIR / "03b_relative_error_boxplots.png"
CDF_FIGURE_PATH = FIGURE_DIR / "03b_mean_error_cdf.png"
P95_FIGURE_PATH = FIGURE_DIR / "03b_p95_error_by_coverage.png"
PROFILE_FIGURE_PATH = FIGURE_DIR / "03b_profile_feature_error.png"

EXPECTED_INTERVALS = 96
COVERAGE_LEVELS = [96, 95, 94, 93, 92]
MASK_LEVELS = [95, 94, 93, 92]
MASK_REPLICATES = 3
RANDOM_SEED = 314159
CSV_CHUNK_SIZE = 250_000
MASK_BATCH_SIZE = 10_000
INTERVAL_HOURS = 0.25


## 1. Validate the frozen upstream inputs

The interval, coverage, complete-day shape, and EDA manifests must agree. The 03A configuration must already be frozen; its target policy is recorded here for provenance, while the masking analysis below uses complete days and does not alter that policy.


In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_csv_atomic(frame, path):
    temporary_path = Path(path).with_suffix(".tmp.csv")
    frame.to_csv(temporary_path, index=False, encoding="utf-8", na_rep="", float_format="%.12g")
    temporary_path.replace(path)

def file_state(path):
    stat = Path(path).stat()
    return {"path": str(Path(path).resolve()), "size_bytes": int(stat.st_size), "modified_ns": int(stat.st_mtime_ns)}

required_inputs = [INTERVAL_PATH, PREPROCESSING_MANIFEST_PATH, COVERAGE_PATH, COMPLETE_DAY_PATH, EDA_MANIFEST_PATH, TARGET_CONFIG_PATH]
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(f"Required validated inputs are missing: {missing_inputs}")

preprocessing_manifest = read_json(PREPROCESSING_MANIFEST_PATH)
eda_manifest = read_json(EDA_MANIFEST_PATH)
target_config = read_json(TARGET_CONFIG_PATH)
if not all(preprocessing_manifest.get("validation", {}).values()):
    raise RuntimeError("Preprocessing manifest contains a failed validation.")
if not all(eda_manifest.get("validation", {}).values()):
    raise RuntimeError("Interval EDA manifest contains a failed validation.")
if target_config.get("status") != "frozen_primary_research_policy":
    raise RuntimeError("03A target policy is not frozen; run/review 03A before 03B.")
if target_config.get("selected_parameters", {}).get("rolling_window_closed") != "left":
    raise RuntimeError("03A policy does not confirm a past-only rolling window.")

INTERVAL_INPUT_STATE_BEFORE = file_state(INTERVAL_PATH)
source_signature = preprocessing_manifest["primary_signature"]
if INTERVAL_INPUT_STATE_BEFORE["size_bytes"] != int(source_signature["size_bytes"]):
    raise RuntimeError("Preprocessed interval file size differs from its validated manifest.")
print(f"Project root: {PROJECT_ROOT}")
print(f"Frozen target configuration: {target_config['configuration_id']}")
print(f"Preprocessed interval input: {INTERVAL_INPUT_STATE_BEFORE['size_bytes']:,} bytes")
print(f"Outputs: {TABLE_DIR}")


## 2. Reconstruct complete days and empirical missing-slot patterns

One interval pass builds a 96-slot value vector for every complete household-day. For incomplete days with exactly 1–4 missing valid slots, it records the actual missing slot positions. These observed patterns are sampled with replacement for each masking replicate and applied to every complete reference day.


In [ ]:
coverage = pd.read_csv(
    COVERAGE_PATH,
    usecols=["household_id", "measurement_date", "valid_interval_count", "coverage_ratio", "is_complete_96_day"],
    dtype={"household_id": "string", "valid_interval_count": "int16", "coverage_ratio": "float32", "is_complete_96_day": "boolean"},
    true_values=["True"], false_values=["False"], parse_dates=["measurement_date"],
    keep_default_na=False, na_values=[""], encoding="utf-8", low_memory=False,
)
coverage["measurement_date"] = pd.to_datetime(coverage["measurement_date"], errors="raise").dt.normalize()
coverage["household_id"] = coverage["household_id"].astype("string")
coverage = coverage.sort_values(["household_id", "measurement_date"], kind="mergesort").reset_index(drop=True)
if coverage.duplicated(["household_id", "measurement_date"]).any():
    raise RuntimeError("Coverage audit has duplicate household-date keys.")
if not coverage["is_complete_96_day"].eq(coverage["valid_interval_count"].eq(96)).all():
    raise RuntimeError("Complete-day coverage flags disagree with valid-slot counts.")

complete = pd.read_csv(
    COMPLETE_DAY_PATH,
    usecols=["household_id", "measurement_date", "valid_interval_count", "daily_mean_kw", "daily_peak_kw", "interval_p95_kw", "peak_hour", "load_factor"],
    dtype={"household_id": "string", "valid_interval_count": "int16", "daily_mean_kw": "float64", "daily_peak_kw": "float64", "interval_p95_kw": "float64", "peak_hour": "int8", "load_factor": "float64"},
    parse_dates=["measurement_date"], keep_default_na=False, na_values=[""], encoding="utf-8", low_memory=False,
)
complete["measurement_date"] = pd.to_datetime(complete["measurement_date"], errors="raise").dt.normalize()
complete["household_id"] = complete["household_id"].astype("string")
complete = complete.sort_values(["household_id", "measurement_date"], kind="mergesort").reset_index(drop=True)
if not complete["valid_interval_count"].eq(96).all() or complete.duplicated(["household_id", "measurement_date"]).any():
    raise RuntimeError("Complete-day EDA reference has invalid counts or duplicate keys.")

households = pd.Index(coverage["household_id"].drop_duplicates().astype(str))
household_code = {household_id: code for code, household_id in enumerate(households)}
coverage_house = coverage["household_id"].astype(str).map(household_code).to_numpy(dtype=np.int32)
coverage_day = coverage["measurement_date"].values.astype("datetime64[D]").astype(np.int32)
coverage_keys = coverage_house.astype(np.int64) * 100_000 + coverage_day.astype(np.int64)
coverage_key_index = pd.Index(coverage_keys, name="household_day_key")
if not coverage_key_index.is_unique:
    raise RuntimeError("Encoded coverage household-date keys are not unique.")

complete_house = complete["household_id"].astype(str).map(household_code).to_numpy(dtype=np.int32)
complete_day = complete["measurement_date"].values.astype("datetime64[D]").astype(np.int32)
complete_coverage_rows = coverage_key_index.get_indexer(complete_house.astype(np.int64) * 100_000 + complete_day.astype(np.int64))
if (complete_coverage_rows < 0).any() or not coverage.loc[complete_coverage_rows, "valid_interval_count"].eq(96).all():
    raise RuntimeError("A complete-day EDA row does not map to complete coverage.")
complete_position_by_coverage = np.full(len(coverage), -1, dtype=np.int32)
complete_position_by_coverage[complete_coverage_rows] = np.arange(len(complete), dtype=np.int32)
complete_slots = np.full((len(complete), EXPECTED_INTERVALS), np.nan, dtype=np.float32)
pattern_candidate = coverage["valid_interval_count"].between(92, 95).to_numpy()
pattern_valid_slots = defaultdict(set)
observed_valid_counts = np.zeros(len(coverage), dtype=np.uint16)
rows_scanned = 0
valid_rows_scanned = 0

reader = pd.read_csv(
    INTERVAL_PATH,
    usecols=["household_id", "measurement_date", "time_slot_index", "avg_import_kw", "is_valid_for_daily_aggregation"],
    dtype={"household_id": "string", "measurement_date": "string", "time_slot_index": "int16", "avg_import_kw": "float32", "is_valid_for_daily_aggregation": "boolean"},
    true_values=["True"], false_values=["False"], chunksize=CSV_CHUNK_SIZE,
    keep_default_na=False, na_values=[""], encoding="utf-8", low_memory=False,
)
for frame in reader:
    rows_scanned += len(frame)
    dates = pd.to_datetime(frame["measurement_date"], errors="raise").values.astype("datetime64[D]").astype(np.int32)
    household_values = frame["household_id"].astype(str)
    codes = household_values.map(household_code)
    if codes.isna().any():
        raise RuntimeError("Interval CSV contains a household absent from coverage audit.")
    day_rows = coverage_key_index.get_indexer(codes.to_numpy(dtype=np.int64) * 100_000 + dates.astype(np.int64))
    if (day_rows < 0).any():
        raise RuntimeError("Interval CSV contains a household-date absent from coverage audit.")
    valid = frame["is_valid_for_daily_aggregation"].fillna(False).to_numpy(dtype=bool)
    if not valid.any():
        continue
    valid_rows_scanned += int(valid.sum())
    valid_day_rows = day_rows[valid].astype(np.int32)
    valid_slots = frame.loc[valid, "time_slot_index"].to_numpy(dtype=np.int16)
    valid_kw = frame.loc[valid, "avg_import_kw"].to_numpy(dtype=np.float32)
    if np.isnan(valid_kw).any() or ((valid_slots < 0) | (valid_slots >= EXPECTED_INTERVALS)).any():
        raise RuntimeError("A valid interval has missing demand or an invalid slot index.")
    observed_valid_counts += np.bincount(valid_day_rows, minlength=len(coverage)).astype(np.uint16)
    complete_positions = complete_position_by_coverage[valid_day_rows]
    keep_complete = complete_positions >= 0
    if keep_complete.any():
        complete_slots[complete_positions[keep_complete], valid_slots[keep_complete]] = valid_kw[keep_complete]
    keep_patterns = pattern_candidate[valid_day_rows]
    if keep_patterns.any():
        pattern_frame = pd.DataFrame({"day_row": valid_day_rows[keep_patterns], "slot": valid_slots[keep_patterns]})
        for day_row, slots in pattern_frame.groupby("day_row", sort=False)["slot"]:
            pattern_valid_slots[int(day_row)].update(int(slot) for slot in slots.tolist())

if rows_scanned != int(source_signature["rows"]):
    raise RuntimeError(f"Scanned {rows_scanned:,} interval rows; manifest expects {int(source_signature['rows']):,}.")
expected_valid_rows = int(preprocessing_manifest["counts"]["valid_aggregation_rows"])
if valid_rows_scanned != expected_valid_rows:
    raise RuntimeError(f"Valid interval count {valid_rows_scanned:,} differs from manifest {expected_valid_rows:,}.")
if not np.array_equal(observed_valid_counts, coverage["valid_interval_count"].to_numpy(dtype=np.uint16)):
    raise RuntimeError("Streamed valid-slot counts disagree with the coverage audit.")
if not np.isfinite(complete_slots).all():
    raise RuntimeError("At least one complete reference day lacks a finite interval value.")

pattern_pools = {level: [] for level in MASK_LEVELS}
for day_row in np.flatnonzero(pattern_candidate):
    level = int(coverage.iloc[day_row]["valid_interval_count"])
    valid_slots = pattern_valid_slots.get(int(day_row), set())
    missing_slots = tuple(slot for slot in range(EXPECTED_INTERVALS) if slot not in valid_slots)
    if len(missing_slots) != EXPECTED_INTERVALS - level:
        raise RuntimeError(f"Reconstructed missing slots disagree with coverage row {day_row}.")
    pattern_pools[level].append(missing_slots)
for level, pool in pattern_pools.items():
    if not pool:
        raise RuntimeError(f"No empirical missing-slot patterns found for {level}/96 days.")

streamed_peaks = complete_slots.max(axis=1)
if not np.allclose(streamed_peaks, complete["daily_peak_kw"].to_numpy(), rtol=0, atol=1e-5):
    raise RuntimeError("Reconstructed complete-day peaks disagree with the EDA reference.")
print(f"Observed household-days: {len(coverage):,}; complete references: {len(complete):,}")
display(pd.DataFrame([{"coverage_level": level, "empirical_pattern_days": len(pool)} for level, pool in pattern_pools.items()]))


## 3. Mask complete days and measure feature error

Each replicate samples one observed pattern per complete reference day at each coverage level. The masked values use the same definitions as the daily feature pipeline: active ratio is the share of valid intervals with positive import, load factor is mean divided by peak, and peak hour is the hour containing the first maximum slot. Peak-time error is circular, so 23:45 to 00:00 is 15 minutes.

For ratio features, absolute error is reported in ratio points. For kW features, both absolute and relative error are reported. Full 96/96 values provide a zero-error baseline.


In [ ]:
def circular_hour_difference_minutes(left_hour, right_hour):
    delta = np.abs(left_hour.astype(np.float64) - right_hour.astype(np.float64))
    return np.minimum(delta, 24.0 - delta) * 60.0

def error_record(feature, coverage_level, replicate, absolute_error, reference, absolute_tolerances=(), relative=True):
    absolute_error = np.asarray(absolute_error, dtype=np.float64)
    reference = np.asarray(reference, dtype=np.float64)
    finite = np.isfinite(absolute_error) & np.isfinite(reference)
    absolute_error = absolute_error[finite]
    reference = reference[finite]
    relative_error = np.divide(absolute_error, np.abs(reference), out=np.full(len(reference), np.nan), where=np.abs(reference) > 0) if relative else np.full(len(reference), np.nan)
    result = {
        "feature": feature, "coverage_level": int(coverage_level), "replicate": int(replicate),
        "n_reference_days": int(len(absolute_error)),
        "mean_absolute_error": float(np.mean(absolute_error)),
        "median_absolute_error": float(np.median(absolute_error)),
        "p90_absolute_error": float(np.quantile(absolute_error, .90)),
        "p95_absolute_error": float(np.quantile(absolute_error, .95)),
        "max_absolute_error": float(np.max(absolute_error)),
        "median_relative_error": float(np.nanmedian(relative_error)) if np.isfinite(relative_error).any() else np.nan,
        "p90_relative_error": float(np.nanquantile(relative_error, .90)) if np.isfinite(relative_error).any() else np.nan,
        "p95_relative_error": float(np.nanquantile(relative_error, .95)) if np.isfinite(relative_error).any() else np.nan,
    }
    for tolerance in [0.01, 0.02, 0.05]:
        result[f"share_relative_error_le_{int(tolerance*100)}pct"] = float(np.nanmean(relative_error <= tolerance)) if np.isfinite(relative_error).any() else np.nan
    for tolerance in absolute_tolerances:
        label = str(tolerance).replace(".", "p")
        result[f"share_absolute_error_le_{label}"] = float(np.mean(absolute_error <= tolerance))
    return result

full = complete_slots.astype(np.float64)
full_mean = full.mean(axis=1)
full_std = full.std(axis=1, ddof=0)
full_peak = full.max(axis=1)
full_peak_slot = full.argmax(axis=1)
full_active = (full > 0).mean(axis=1)
full_load_factor = np.divide(full_mean, full_peak, out=np.full(len(full), np.nan), where=full_peak > 0)
full_p95 = np.quantile(full, .95, axis=1, method="linear")
full_peak_hour = (full_peak_slot // 4).astype(np.float64)
full_peak_time_minutes = full_peak_slot.astype(np.float64) * 15.0
full_energy = full.sum(axis=1) * INTERVAL_HOURS

reference_metrics = {
    "daily_peak_kw": full_peak, "daily_mean_kw": full_mean, "daily_std_kw": full_std,
    "import_active_ratio": full_active, "load_factor": full_load_factor,
    "interval_p95_kw": full_p95, "peak_hour": full_peak_hour,
    "peak_time_minutes": full_peak_time_minutes, "daily_import_energy_kwh": full_energy,
}
raw_records = []
masked_feature_samples = []
for level in MASK_LEVELS:
    patterns = np.asarray(pattern_pools[level], dtype=np.int16)
    for replicate in range(1, MASK_REPLICATES + 1):
        rng = np.random.default_rng(RANDOM_SEED + level * 1000 + replicate)
        sampled_pattern_ids = rng.integers(0, len(patterns), size=len(full))
        sampled_patterns = patterns[sampled_pattern_ids]
        masked_metrics = {name: [] for name in reference_metrics}
        for start in range(0, len(full), MASK_BATCH_SIZE):
            stop = min(start + MASK_BATCH_SIZE, len(full))
            batch = full[start:stop].copy()
            row_index = np.arange(stop - start)
            batch[row_index[:, None], sampled_patterns[start:stop]] = np.nan
            mean = np.nanmean(batch, axis=1)
            std = np.nanstd(batch, axis=1, ddof=0)
            peak = np.nanmax(batch, axis=1)
            peak_slot = np.nanargmax(batch, axis=1)
            active = np.nanmean(batch > 0, axis=1)
            load_factor = np.divide(mean, peak, out=np.full(len(mean), np.nan), where=peak > 0)
            p95 = np.nanquantile(batch, .95, axis=1, method="linear")
            energy = np.nansum(batch, axis=1) * INTERVAL_HOURS
            values = {
                "daily_peak_kw": peak, "daily_mean_kw": mean, "daily_std_kw": std,
                "import_active_ratio": active, "load_factor": load_factor,
                "interval_p95_kw": p95, "peak_hour": (peak_slot // 4).astype(np.float64),
                "peak_time_minutes": peak_slot.astype(np.float64) * 15.0, "daily_import_energy_kwh": energy,
            }
            for name, value in values.items():
                masked_metrics[name].append(value)
        masked_metrics = {name: np.concatenate(parts) for name, parts in masked_metrics.items()}
        for name, reference in reference_metrics.items():
            candidate = masked_metrics[name]
            if name == "peak_hour":
                absolute_error = circular_hour_difference_minutes(candidate, reference)
                raw_records.append(error_record(name, level, replicate, absolute_error, np.full(len(reference), 1.0), absolute_tolerances=[60, 120, 240], relative=False))
            elif name == "peak_time_minutes":
                delta = np.abs(candidate - reference)
                absolute_error = np.minimum(delta, 1440.0 - delta)
                raw_records.append(error_record(name, level, replicate, absolute_error, np.ones(len(reference)), absolute_tolerances=[15, 30, 60], relative=False))
            elif name == "import_active_ratio":
                absolute_error = np.abs(candidate - reference)
                raw_records.append(error_record(name, level, replicate, absolute_error, reference, absolute_tolerances=[.01, .02, .05], relative=False))
            else:
                absolute_error = np.abs(candidate - reference)
                raw_records.append(error_record(name, level, replicate, absolute_error, reference, absolute_tolerances=[.01, .05, .10], relative=True))
        # Save a deterministic subset only for plotting; the full paired arrays are summarized above.
        sample_ix = np.linspace(0, len(full) - 1, min(2500, len(full)), dtype=np.int64)
        for name in ["daily_mean_kw", "daily_peak_kw", "interval_p95_kw", "load_factor"]:
            ref = reference_metrics[name][sample_ix]
            cand = masked_metrics[name][sample_ix]
            rel = np.divide(np.abs(cand - ref), np.abs(ref), out=np.full(len(ref), np.nan), where=np.abs(ref) > 0)
            masked_feature_samples.extend({"feature": name, "coverage_level": level, "replicate": replicate, "relative_error": float(err)} for err in rel if np.isfinite(err))

# 96/96 reference rows have zero error by construction.
for name, reference in reference_metrics.items():
    if name == "peak_hour":
        raw_records.append(error_record(name, 96, 0, np.zeros(len(reference)), np.ones(len(reference)), absolute_tolerances=[60, 120, 240], relative=False))
    elif name == "peak_time_minutes":
        raw_records.append(error_record(name, 96, 0, np.zeros(len(reference)), np.ones(len(reference)), absolute_tolerances=[15, 30, 60], relative=False))
    elif name == "import_active_ratio":
        raw_records.append(error_record(name, 96, 0, np.zeros(len(reference)), reference, absolute_tolerances=[.01, .02, .05], relative=False))
    else:
        raw_records.append(error_record(name, 96, 0, np.zeros(len(reference)), reference, absolute_tolerances=[.01, .05, .10], relative=True))

raw_summary = pd.DataFrame(raw_records)
display(raw_summary.loc[raw_summary["coverage_level"].isin([96, 94, 92]), ["feature", "coverage_level", "replicate", "median_absolute_error", "p95_absolute_error", "p95_relative_error"]].head(30))


## 4. Summarize feature-specific reliability and energy behavior

The tables retain each replicate’s error profile and report the replicate mean and spread. Candidate minimum coverage is screened feature by feature using declared tolerances: P95 relative error ≤5% for kW/load-factor features, P95 absolute ratio error ≤0.02 for active-import ratio, and P95 circular peak-hour error ≤60 minutes. These are transparent screening criteria, not universal scientific standards; the policy remains provisional for researcher review.

Energy is evaluated without scaling partial-day sums. A near-perfect mean/energy relationship on complete days supports omitting daily energy as a separate lag predictor because it is a 24-hour rescaling of mean power.


In [ ]:
summary_metrics = [
    "mean_absolute_error", "median_absolute_error", "p90_absolute_error", "p95_absolute_error", "max_absolute_error",
    "median_relative_error", "p90_relative_error", "p95_relative_error",
    "share_relative_error_le_1pct", "share_relative_error_le_2pct", "share_relative_error_le_5pct",
    "share_absolute_error_le_0p01", "share_absolute_error_le_0p02", "share_absolute_error_le_0p05",
    "share_absolute_error_le_15", "share_absolute_error_le_30", "share_absolute_error_le_60",
]
summary_parts = []
for (feature, level), group in raw_summary.groupby(["feature", "coverage_level"], sort=False):
    row = {"feature": feature, "coverage_level": int(level), "n_replicates": int(len(group)), "n_reference_days_per_replicate": int(group["n_reference_days"].min())}
    for metric in summary_metrics:
        if metric in group:
            row[metric + "_replicate_mean"] = float(group[metric].mean())
            row[metric + "_replicate_sd"] = float(group[metric].std(ddof=1)) if len(group) > 1 else 0.0
    summary_parts.append(row)
feature_summary = pd.DataFrame(summary_parts).sort_values(["feature", "coverage_level"], ascending=[True, False], kind="mergesort").reset_index(drop=True)

energy_pair = pd.DataFrame({"daily_mean_kw": full_mean, "daily_import_energy_kwh": full_energy})
energy_correlation = float(energy_pair.corr().iloc[0, 1])
energy_ratio = full_energy / np.where(full_mean != 0, full_mean, np.nan)
energy_audit = pd.DataFrame([{
    "complete_reference_days": int(len(full)),
    "mean_energy_correlation": energy_correlation,
    "median_energy_to_mean_ratio_hours": float(np.nanmedian(energy_ratio)),
    "energy_is_24h_rescaling": bool(abs(energy_correlation) >= 0.999999 and np.isclose(np.nanmedian(energy_ratio), 24.0, rtol=.01, atol=.01)),
    "partial_energy_adjustment_used": False,
    "feature_guidance": "Omit redundant daily-energy lag if it is a 24-hour rescaling of mean power; do not extrapolate incomplete-day energy." if abs(energy_correlation) >= 0.999999 else "Keep energy separate for researcher review; partial-day energy remains unscaled.",
}])

# Transparent candidate thresholds; choose the least restrictive tested coverage that passes.
criteria = {
    "daily_peak_kw": ("p95_relative_error_replicate_mean", .05, "P95 relative error <=5%"),
    "daily_mean_kw": ("p95_relative_error_replicate_mean", .05, "P95 relative error <=5%"),
    "daily_std_kw": ("p95_relative_error_replicate_mean", .05, "P95 relative error <=5%"),
    "load_factor": ("p95_relative_error_replicate_mean", .05, "P95 relative error <=5%"),
    "interval_p95_kw": ("p95_relative_error_replicate_mean", .05, "P95 relative error <=5%"),
    "import_active_ratio": ("p95_absolute_error_replicate_mean", .02, "P95 absolute ratio error <=0.02"),
    "peak_time_minutes": ("p95_absolute_error_replicate_mean", 60.0, "P95 circular timing error <=60 minutes"),
}
policy_rows = []
for feature, (metric, tolerance, criterion) in criteria.items():
    values = feature_summary.loc[feature_summary["feature"].eq(feature)].set_index("coverage_level")
    passing = []
    for level in [92, 93, 94, 95, 96]:
        measured = values.loc[level, metric] if level in values.index and metric in values.columns else np.nan
        if pd.notna(measured) and measured <= tolerance:
            passing.append(level)
    policy_rows.append({
        "feature": feature, "screening_metric": metric, "screening_tolerance": tolerance,
        "criterion": criterion, "candidate_minimum_coverage": min(passing) if passing else pd.NA,
        "passing_tested_coverages": ",".join(map(str, passing)),
        "policy_status": "provisional_candidate_researcher_review_required",
        "decision_note": "Candidate is based on masking error only; no model score used." if passing else "No tested partial-coverage level passes; retain complete-day values pending review.",
    })
policy_candidates = pd.DataFrame(policy_rows)

print("Feature masking summary")
display(feature_summary)
print("Provisional feature-specific screening candidates")
display(policy_candidates)
print("Energy diagnostic")
display(energy_audit)


## 5. Visual evidence

The plots show relative error distributions for kW/load-factor features, mean-error CDFs, the P95 error trend across coverage levels, and a combined profile-feature error view. The 96/96 baseline is omitted from the boxplot and shown as zero in the summary table.


In [ ]:
sample_frame = pd.DataFrame(masked_feature_samples)
plot_order = ["daily_mean_kw", "daily_peak_kw", "interval_p95_kw", "load_factor"]
fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(data=sample_frame, x="coverage_level", y="relative_error", hue="feature", order=[92, 93, 94, 95], hue_order=plot_order, showfliers=False, ax=ax)
ax.set_title("Relative feature error after empirical interval masking")
ax.set_xlabel("Retained valid intervals out of 96")
ax.set_ylabel("Absolute relative error")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
ax.legend(title="Feature", frameon=True)
fig.tight_layout()
fig.savefig(BOX_FIGURE_PATH, dpi=170, bbox_inches="tight")
display(fig)
plt.close(fig)

# CDF for masked daily mean errors: all sampled reference days are pooled across replicates.
fig, ax = plt.subplots(figsize=(9, 5.5))
for level in [92, 93, 94, 95]:
    g = sample_frame.loc[sample_frame["feature"].eq("daily_mean_kw") & sample_frame["coverage_level"].eq(level), "relative_error"].dropna().to_numpy()
    g = np.sort(g)
    if len(g):
        ax.plot(g, np.arange(1, len(g) + 1) / len(g), label=f"{level}/96")
ax.axvline(.01, color="gray", linestyle=":", label="1% error")
ax.axvline(.02, color="#f58518", linestyle="--", label="2% error")
ax.axvline(.05, color="#d62728", linestyle="--", label="5% error")
ax.set_title("Empirical CDF of daily mean relative error")
ax.set_xlabel("Absolute relative error")
ax.set_ylabel("Share of masked days at or below error")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
ax.set_xlim(left=0)
ax.set_ylim(0, 1.01)
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(CDF_FIGURE_PATH, dpi=170, bbox_inches="tight")
display(fig)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5.5))
for feature, label in [("daily_mean_kw", "Mean"), ("daily_peak_kw", "Peak"), ("interval_p95_kw", "Interval P95"), ("load_factor", "Load factor")]:
    g = feature_summary.loc[feature_summary["feature"].eq(feature)].sort_values("coverage_level")
    ax.plot(g["coverage_level"], g["p95_relative_error_replicate_mean"], marker="o", label=label)
ax.set_title("95th percentile relative error by coverage level")
ax.set_xlabel("Retained valid intervals out of 96")
ax.set_ylabel("P95 relative error")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
ax.set_xticks([92, 93, 94, 95, 96])
ax.invert_xaxis()
ax.legend()
fig.tight_layout()
fig.savefig(P95_FIGURE_PATH, dpi=170, bbox_inches="tight")
display(fig)
plt.close(fig)

profile_features = ["import_active_ratio", "load_factor", "interval_p95_kw", "peak_time_minutes"]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, feature in zip(axes.flat, profile_features):
    g = feature_summary.loc[feature_summary["feature"].eq(feature)].sort_values("coverage_level")
    metric = "p95_absolute_error_replicate_mean" if feature in ["import_active_ratio", "peak_time_minutes"] else "p95_relative_error_replicate_mean"
    ax.plot(g["coverage_level"], g[metric], marker="o", color="#4c78a8")
    title = {"import_active_ratio": "Active import ratio", "load_factor": "Load factor", "interval_p95_kw": "Interval P95", "peak_time_minutes": "Peak time"}[feature]
    ax.set_title(title)
    ax.set_xlabel("Retained valid intervals out of 96")
    ax.set_ylabel("P95 absolute error" if feature in ["import_active_ratio", "peak_time_minutes"] else "P95 relative error")
    ax.set_xticks([92, 93, 94, 95, 96])
    ax.invert_xaxis()
    if feature == "import_active_ratio":
        ax.axhline(.02, color="#d62728", linestyle="--", linewidth=1)
    elif feature == "peak_time_minutes":
        ax.axhline(60, color="#d62728", linestyle="--", linewidth=1)
    else:
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
fig.suptitle("Profile feature stability under empirical masking", fontweight="bold")
fig.tight_layout()
fig.savefig(PROFILE_FIGURE_PATH, dpi=170, bbox_inches="tight")
display(fig)
plt.close(fig)


## 6. Save evidence and validate the run

The manifest records the exact source signatures, frozen 03A policy, mask levels, replicate count, seed, and screening criteria. The validation table checks source reconciliation, pattern availability, metric coverage, candidate policy labeling, energy handling, and output integrity.


In [ ]:
write_csv_atomic(feature_summary, SUMMARY_PATH)
write_csv_atomic(policy_candidates, POLICY_PATH)
write_csv_atomic(energy_audit, ENERGY_PATH)

# Freeze the already-applied masking-screen candidates; do not retune them from model results.
frozen_cutoffs = {
    str(row.feature): int(row.candidate_minimum_coverage)
    for row in policy_candidates.itertuples(index=False)
}
expected_cutoffs = {
    "daily_peak_kw": 92, "daily_mean_kw": 93, "daily_std_kw": 93,
    "load_factor": 93, "interval_p95_kw": 95, "import_active_ratio": 95,
    "peak_time_minutes": 92,
}
if frozen_cutoffs != expected_cutoffs:
    raise RuntimeError(f"03B evidence candidates changed from the policy already used downstream: {frozen_cutoffs}")
rationales = {
    "daily_peak_kw": "Minimum tested coverage passing the P95 relative-error tolerance of 5%.",
    "daily_mean_kw": "Minimum tested coverage passing the P95 relative-error tolerance of 5%.",
    "daily_std_kw": "Minimum tested coverage passing the P95 relative-error tolerance of 5%.",
    "load_factor": "Minimum tested coverage passing the P95 relative-error tolerance of 5%.",
    "interval_p95_kw": "Minimum tested coverage passing the P95 relative-error tolerance of 5%; stricter support retained because lower levels did not pass.",
    "import_active_ratio": "Minimum tested coverage passing the P95 absolute-ratio-error tolerance of 0.02.",
    "peak_time_minutes": "Minimum tested coverage passing the P95 circular-timing-error tolerance of 60 minutes.",
}
frozen_config = {
    "configuration_version": 1,
    "configuration_id": "FQ_PEAK92_MEAN93_STD93_LOAD93_IP95_95_ACTIVE95_TIME92_v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "status": "frozen_feature_quality_policy",
    "upstream_target_configuration_id": target_config["configuration_id"],
    "minimum_valid_intervals": frozen_cutoffs,
    "daily_energy_policy": "omit_from_predictors",
    "decision_rationales": rationales,
    "decision_basis": "03B empirical masking stability criteria only; no downstream model score or final-test metric used.",
    "evidence": {
        "candidate_table": str(POLICY_PATH.resolve()),
        "masking_summary": str(SUMMARY_PATH.resolve()),
        "energy_audit": str(ENERGY_PATH.resolve()),
    },
}
temporary_config = CONFIG_PATH.with_suffix(".tmp.json")
temporary_config.write_text(json.dumps(frozen_config, indent=2, allow_nan=False), encoding="utf-8")
temporary_config.replace(CONFIG_PATH)

input_state_after = file_state(INTERVAL_PATH)
input_unchanged = input_state_after == INTERVAL_INPUT_STATE_BEFORE
essential_summary_columns = ["mean_absolute_error_replicate_mean", "median_absolute_error_replicate_mean", "p90_absolute_error_replicate_mean", "p95_absolute_error_replicate_mean"]
summary_finite = bool(np.isfinite(feature_summary[essential_summary_columns].to_numpy(dtype=np.float64)).all())
coverage_levels_complete = set(feature_summary["coverage_level"].astype(int)) == {92, 93, 94, 95, 96}
features_complete = set(criteria).issubset(set(feature_summary["feature"]))
all_candidates_reviewable = True
energy_not_extrapolated = not bool(energy_audit["partial_energy_adjustment_used"].iloc[0])
artifacts_exist = all(path.exists() and path.stat().st_size > 0 for path in [SUMMARY_PATH, POLICY_PATH, ENERGY_PATH, CONFIG_PATH, BOX_FIGURE_PATH, CDF_FIGURE_PATH, P95_FIGURE_PATH, PROFILE_FIGURE_PATH])

validation = pd.DataFrame([
    ("frozen_upstream_target_policy", target_config.get("status") == "frozen_primary_research_policy", target_config.get("configuration_id", "")),
    ("interval_rows_match_preprocessing_manifest", rows_scanned == int(source_signature["rows"]), f"{rows_scanned:,}"),
    ("valid_interval_rows_match_preprocessing_manifest", valid_rows_scanned == expected_valid_rows, f"{valid_rows_scanned:,}"),
    ("coverage_counts_reconciled", np.array_equal(observed_valid_counts, coverage["valid_interval_count"].to_numpy(dtype=np.uint16)), f"{len(coverage):,} household-days"),
    ("complete_reference_vectors_reconstructed", bool(np.isfinite(complete_slots).all()) and len(complete) == int(coverage["is_complete_96_day"].sum()), f"{len(complete):,} complete days × 96 slots"),
    ("complete_reference_peaks_match_eda", bool(np.allclose(streamed_peaks, complete["daily_peak_kw"].to_numpy(), rtol=0, atol=1e-5)), "streamed max compared to complete-day EDA"),
    ("empirical_patterns_available_for_92_to_95", all(len(pattern_pools[level]) > 0 for level in MASK_LEVELS), str({level: len(pool) for level, pool in pattern_pools.items()})),
    ("all_feature_coverage_levels_reported", coverage_levels_complete, str(sorted(feature_summary["coverage_level"].unique().tolist()))),
    ("required_predictor_features_reported", features_complete, str(sorted(set(criteria) - set(feature_summary["feature"]))),),
    ("summary_metrics_finite", summary_finite, f"{len(feature_summary):,} feature/coverage rows"),
    ("frozen_policy_matches_previously_applied_candidates", frozen_config["status"] == "frozen_feature_quality_policy" and frozen_cutoffs == expected_cutoffs, f"{frozen_config['configuration_id']}: {frozen_cutoffs}"),
    ("partial_energy_not_extrapolated", energy_not_extrapolated, "masked energy uses observed intervals only"),
    ("interval_input_unchanged", input_unchanged, str(input_state_after)),
    ("expected_output_artifacts_present", artifacts_exist, str(TABLE_DIR)),
], columns=["check", "passed", "evidence"])
write_csv_atomic(validation, VALIDATION_PATH)
if not validation["passed"].all():
    raise RuntimeError(f"03B validation failed: {validation.loc[~validation['passed'], 'check'].tolist()}")

manifest = {
    "notebook": "03b_historical_feature_quality_experiments.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "status": "frozen_feature_quality_policy",
    "feature_quality_configuration_id": frozen_config["configuration_id"],
    "feature_quality_configuration": frozen_config,
    "upstream_target_configuration_id": target_config["configuration_id"],
    "upstream_target_policy": target_config["selected_parameters"],
    "input_signatures": {
        "preprocessed_intervals": {"path": str(INTERVAL_PATH.resolve()), "size_bytes": INTERVAL_INPUT_STATE_BEFORE["size_bytes"], "rows": int(source_signature["rows"])},
        "coverage_audit": {"path": str(COVERAGE_PATH.resolve()), "rows": int(len(coverage))},
        "complete_day_eda": {"path": str(COMPLETE_DAY_PATH.resolve()), "rows": int(len(complete))},
    },
    "experiment": {
        "reference": "genuine 96/96 household-days",
        "empirical_mask_levels": MASK_LEVELS,
        "empirical_pattern_source_days": {str(level): int(len(pattern_pools[level])) for level in MASK_LEVELS},
        "replicates_per_level": MASK_REPLICATES,
        "random_seed_base": RANDOM_SEED,
        "screening_criteria": {feature: {"metric": metric, "tolerance": tolerance, "description": description} for feature, (metric, tolerance, description) in criteria.items()},
        "energy_extrapolation": False,
    },
    "population": {"complete_reference_days": int(len(complete)), "households": int(coverage["household_id"].nunique())},
    "outputs": {"summary": str(SUMMARY_PATH.resolve()), "policy_candidates": str(POLICY_PATH.resolve()), "selected_config": str(CONFIG_PATH.resolve()), "energy_audit": str(ENERGY_PATH.resolve()), "validation": str(VALIDATION_PATH.resolve())},
    "validation": {row.check: bool(row.passed) for row in validation.itertuples(index=False)},
}
temporary_manifest = MANIFEST_PATH.with_suffix(".tmp.json")
temporary_manifest.write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding="utf-8")
temporary_manifest.replace(MANIFEST_PATH)

display(validation)
print(f"Saved feature-quality evidence to {TABLE_DIR}")
print(f"Saved figures to {FIGURE_DIR}")


## Interpretation and handoff

The frozen feature-quality configuration records the exact thresholds already applied by feature engineering and the masking-evidence rationale for each. These thresholds were selected from the predeclared stability criteria, without using model scores or final-test metrics. The masking experiment measures within-day error on complete reference days; it does not establish that missingness is random or that a feature is predictive.

Feature engineering must load this frozen configuration and fail if its status or ID is unexpected. Daily energy remains omitted from predictors; incomplete energy values are never scaled to a full day.
